# 🌊 Topic 06: PySpark Structured Streaming & Real-Time Processing

## 1. Structured Streaming Paradigm
Structured Streaming treats live data streams as an **unbounded continuous table**.
- **Micro-Batch Processing:** Small batched queries processed periodically (e.g. every 1 second).
- **Watermarking:** Sets a threshold for how late data can arrive before being discarded from state.

---

## 2. Hands-on: Streaming Aggregations with In-Memory Memory Stream


In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import time

spark = SparkSession.builder.master("local[*]").appName("Structured_Streaming_Demo").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Create a rate stream source (generates continuous rows)
rate_stream_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

# Transform stream: Calculate running count and timestamp window
transformed_stream = rate_stream_df.withColumn("is_even", (F.col("value") % 2 == 0)) \
    .groupBy("is_even") \
    .agg(F.count("value").alias("count"), F.max("timestamp").alias("latest_event"))

# Start streaming query outputting to in-memory sink
query = transformed_stream.writeStream \
    .format("memory") \
    .queryName("streaming_metrics") \
    .outputMode("complete") \
    .start()

print("🌊 Structured Stream Running for 6 seconds...")
time.sleep(6)

print("\n📊 In-Memory Stream Output (Live Aggregations):")
spark.sql("SELECT * FROM streaming_metrics").show()

# Stop streaming query
query.stop()
print("⏹️ Stream Query Stopped.")
